## Module 4: Evaluation

Evaluation is the process of measuring how well a search, RAG, or agent system performs using objective metrics instead of manual testing. It allows us to compare different search methods, prompts, models, and parameter settings. The module follows an A → Q → A'* pattern, where an LLM generates questions from known answers to create a test dataset with expected outputs. Search evaluation checks whether the correct documents are retrieved, while RAG evaluation measures the quality of generated answers, and agent evaluation also considers tool usage. Rather than the online evaluation where the system collects feedback from real users in production, this module mainly focuses on offline evaluation using synthetic data as a starting point, and introduces metrics such as Hit Rate, Mean Reciprocal Rank (MRR), and LLM-as-a-judge to evaluate system performance.

### Part 1: Search Evaluation
This part creates a ground truth dataset and uses it to evaluate retrieval quality.

#### _Section 1: Generating Ground Truth Data_
To evaluate a search system, we need a ground truth dataset that links each query to its correct document. This dataset can be created by human annotators, collected from real user queries, or generated synthetically with an LLM. In this module, an LLM is used to generate several questions for each FAQ document, making the original document the known correct answer for those questions. This synthetic dataset is then used to test whether the search system successfully retrieves the relevant document.

In [4]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"

!wget {PREFIX}/01-agentic-rag/code/ingest.py
!wget {PREFIX}/01-agentic-rag/code/rag_helper.py
!wget {PREFIX}/04-evaluation/code/evaluation_utils.py

--2026-07-12 20:17:14--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 738 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     738  --.-KB/s    in 0s      

2026-07-12 20:17:14 (36.3 MB/s) - ‘ingest.py’ saved [738/738]

--2026-07-12 20:17:14--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1

In [5]:
# load the FAQ data
from ingest import load_faq_data
documents = load_faq_data()

In [6]:
# generate only for llm-zoomcamp course, since full set would take too long
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

print(f"Total llm-zoomcamp pages: {len(documents_llm)}")

# set llm-zoomcamp documents as the main documents to use for evaluation
documents = documents_llm

doc = documents[0]
print(f"Document ID: {doc['id']}")
print(f"Question: {doc['question']}")
print(f"Answer: {doc['answer']}")

Total llm-zoomcamp pages: 113
Document ID: 74eb249bbf
Question: I just discovered the course. Can I still join?
Answer: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


Each document needs a stable, unique ID because it serves as the label in the ground truth dataset. Since questions are generated from a specific document, its ID identifies the correct answer, allowing search evaluation to verify whether the system retrieved the right document.

An LLM is used to generate questions for each document using structured output, which returns data in a predefined format instead of free-form text. By defining the expected structure with a Pydantic model, the generated questions are returned as a consistent list of strings that can be easily processed by code without manual text parsing.

In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

# define instructions for the llm
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
# make sure .env file is loaded (working directory is important here)
from dotenv import load_dotenv
print(load_dotenv())

# check whether the OpenAI client works
from openai import OpenAI
openai_client = OpenAI()

True


Until now we called `responses.create` and read `response.output_text`. For structured output we switch to `responses.parse` and pass `text_format=Questions`, which tells the API to return our class instead of free text.

In [10]:
import json

# prepare the document as json
user_prompt = json.dumps(doc)

# create the messages
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

# call the model
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

result = response.output_parsed
print(result)

# see the list of questions generated
print(result.questions)

questions=['Is it still okay to join this course if I found it late?', 'Can I start the course now even though it already began?', 'If I join late, will I still be able to get a certificate?', 'Do I need to submit my project before submissions close to receive the certificate?', 'What’s the deadline for project submission if I want the course certificate?']
['Is it still okay to join this course if I found it late?', 'Can I start the course now even though it already began?', 'If I join late, will I still be able to get a certificate?', 'Do I need to submit my project before submissions close to receive the certificate?', 'What’s the deadline for project submission if I want the course certificate?']


In [11]:
# import the helper file including reusable written functions
from evaluation_utils import llm_structured

# use the function call on the same document
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Is it too late to join the course if I found it only now?', 'Can new students still sign up for the course after it has already started?', 'If I join late, do I still have a chance to get a certificate?', 'What do I need to do to qualify for a certificate if I start the course now?', 'Are project submissions still open for people who discovered the course recently?']


In [ ]:
# track token usage
usage.input_tokens, usage.output_tokens

(207, 93)

In [16]:
# use helper function to calculate the cost of the request
from evaluation_utils import calc_price

cost = calc_price(usage)
cost

{'input_cost': 0.00015525, 'output_cost': 0.0004185, 'total_cost': 0.00057375}

Each record has two fields:
- question: the question generated by the LLM
- document: the ID of the FAQ document that should answer the question

The document field connects the generated question to the document that contains the answer. Later, when we evaluate search, we'll ask the search engine the generated question. Then we'll check if it retrieves the document with this ID.

In [17]:
# convert the generated questions into ground truth records
records = []
for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Is it too late to join the course if I found it only now?',
  'document': '74eb249bbf'},
 {'question': 'Can new students still sign up for the course after it has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, do I still have a chance to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for a certificate if I start the course now?',
  'document': '74eb249bbf'},
 {'question': 'Are project submissions still open for people who discovered the course recently?',
  'document': '74eb249bbf'}]

### Part 2: RAG and Agent Evaluation
This part evaluates answer quality after retrieval. It also shows the basic idea of agent evaluation: save the final answer and the tool-call trajectory.